# 🔐 WE3 — Differential Privacy
## Training on people's data *without* giving them away

> **The story.** You are the **machine-learning lead at a bank.** You want a model that flags
> clients likely to **default on a loan** — trained on real client records that include
> **sensitive attributes** (income, debts, spending history). To make the target sharp, the model
> is trained *only* on clients with a **past problematic spending history.** That is exactly what
> makes it dangerous: if the trained model quietly reveals that **Alice was in its training set**,
> it has also revealed something Alice never agreed to share.
>
> Before you ship anything, you want a *guarantee*: the model's behaviour should be **almost the
> same whether or not any single person was included**. That guarantee has a name — **Differential
> Privacy** — and by the end of this notebook you will be able to *build* it, from a one-line
> counting query all the way up to **DP-SGD**, the private training algorithm.

**🧭 No prior knowledge assumed.** We build every idea from scratch — *mechanism*, *neighbourhood*,
*sensitivity*, *the Laplace and Gaussian distributions*, *composition* — in plain language, the
moment we need it. If a symbol is new, we unpack it together.

**How this notebook works**
- Short explanations, then small hands-on tasks marked **🎯** for you to complete.
- Interactive **quizzes** and **diagrams** build the intuition before the code.
- Everything runs on **toy numbers** — a real DP training run needs a data-centre; the *ideas* fit
  in a few lines of NumPy. The story is the wrapper; the maths is what you take home.


## 0. Setup

This notebook is **self-contained**: the first cell pulls the exercise files (the `dp_viz.py`
display helpers) directly from the course repository. Run the setup cells below in order.

> 🔑 **While the course repo is private** (testing phase) you need a GitHub access token:
> open the **Secrets** panel (🔑 icon in the left sidebar), add a secret named
> **`GITHUB_TOKEN`**, paste your token, and toggle *Notebook access* on. Once the repo is
> public, no token is needed.

**0.1 — Fetch the exercise files.**

In [ ]:
import os, sys

REPO_OWNER = "eth-fdd-fs26"
REPO_NAME  = "FDD-WE3-private"

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if _in_colab():
    token = ""
    try:                                  # private repo (testing): read token from Secrets
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN") or ""
    except Exception:                     # public repo (students): no token needed
        token = ""
    auth = f"{token}@" if token else ""
    url = f"https://{auth}github.com/{REPO_OWNER}/{REPO_NAME}.git"
    if not os.path.isdir(REPO_NAME):
        print("Cloning the exercise repo…")
        !git clone -q "$url"
    else:                                 # already cloned earlier — refresh to the latest version
        print("Updating the exercise repo to the latest version…")
        !git -C "$REPO_NAME" pull -q "$url" || echo "  (could not pull — using the existing copy)"

# Move to the REPO ROOT — the folder holding the `exercises/` helpers — so imports resolve cleanly.
for _root in [REPO_NAME, ".", os.path.dirname(os.getcwd()), os.getcwd()]:
    if os.path.exists(os.path.join(_root, "exercises", "dp_viz.py")):
        os.chdir(_root)
        break
else:
    raise FileNotFoundError(
        "Could not find the repo (exercises/dp_viz.py). If it is still private, add a "
        "GITHUB_TOKEN secret (see the note above) and re-run this cell.")
sys.path.insert(0, os.path.join(os.getcwd(), "exercises"))   # make the helpers importable
print("Working directory:", os.getcwd())

**0.2 — Install dependencies.** All of these are already on Colab; this just pins versions
(and makes the notebook work outside Colab too).

In [ ]:
%pip install -q -r exercises/requirements_we3.txt

**0.3 — Import the libraries.** The diagrams, quizzes, and plots live in **`dp_viz`** so the
teaching cells stay about the *idea*, not about HTML.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import importlib
import dp_viz
importlib.reload(dp_viz)   # pick up the latest helpers even if a stale copy was cached

np.set_printoptions(precision=3, suppress=True)
print("Environment ready ✅")

---
# Part 1 — The threat: what "leaking a person" means

Before defending anything, let's be precise about the attack we're worried about. The visual below
explains it in plain words — no coding background needed.

In [ ]:
dp_viz.membership_inference_demo()

So the danger isn't that the model is *wrong* — it's that the model, or a released statistic,
**depends too much on any one person**, so that comparing "with them" vs "without them" gives them
away.

### 🧠 Quick check — what is a membership inference attack?

In [ ]:
dp_viz.mc_quiz("membership")

---
# Part 2 — Differential Privacy, built around one client: James

Let's make the danger concrete. Suppose the bank wants to **publicly disclose a single number: how
many of its clients are defaulters.** Sounds harmless. Meet the four clients — and watch how that one
number can betray **James**.

In [ ]:
dp_viz.dp_setup_diagram()

### If we publish the *exact* number, James is cooked
The other three clients conspire: they know their own statuses sum to **2** defaulters, so they just
subtract. With **no noise**, the published count is either 2 or 3 — and that single bit tells them
James's status exactly. The only escape is to make the output a **distribution** that looks the same
in both of James's worlds.

In [ ]:
dp_viz.why_noise_viz()

### So we add noise — but *how much*? What does the attacker actually see?
Say the bank publishes **2.6**. The attacker now compares: *how likely is 2.6 if James repays vs. if
James defaults?* The answer depends entirely on the amount of noise.

In [ ]:
dp_viz.observe_scenarios_viz()

**Low noise leaks; high noise hides but can print nonsense.** That tension is the whole game,
and the next parts turn it into a dial. First, two loose ends this raises.

### From "one observed value" to "any question the attacker asks"
We just reasoned about a single observed value (2.6). But the attacker could instead ask about a
whole **interval** — *"is the number between 2 and 5?"* That set of outputs is the **outcome set S**,
and DP must hold for *every* such S. Happily, the pointwise picture already covers all of them:

In [ ]:
dp_viz.pointwise_equivalence_card()

Here is a real interval `S` drawn on the two worlds — the attacker's *"is it between 2.5 and
6?"* question — with the two probabilities it compares:

In [ ]:
dp_viz.sum_dp_illustration()

### Now name the dial: ε
Everything above says *low noise → easy to expose James; high noise → well hidden*. **ε** turns that
into a number you choose up front — a **leak budget**.

In [ ]:
dp_viz.eps_definition_card()

And here is that `ε` leak budget drawn as a concrete tolerance on a discrete distribution: each
bar is one outcome's probability in one world, and the green window is how far the *other* world's
probability may sit from it — about `±ε`.

In [ ]:
dp_viz.eps_interval_viz(0.2)

### 🧠 Quick check — the setup
Click every statement you think is **true**.

In [ ]:
dp_viz.true_false_quiz("dp_setup")

### The neighbourhood is *your* choice
The definition keeps saying "for all neighbouring (a, a′)". **You** get to define what neighbouring
means — and that choice is exactly *what you are hiding* (here: James's status).

In [ ]:
dp_viz.neighbourhood_examples()

---
# Part 3 — The noise itself: Laplace, Gaussian, and ε on the curves

We now know *why* we add noise and what ε buys us. Time to look at the noise we actually add. Two
distributions do almost all the work in DP — the **Laplace** and the **Gaussian**:

In [ ]:
dp_viz.laplace_gaussian_shapes()

### Seeing ε directly on the two worlds
Recall the *leak budget*: ε caps the factor `e^ε` by which the two worlds' probabilities may differ.
Watch the **same** mechanism at a **loose** budget and a **tight** one — the bottom panel shows the
pointwise ratio is always trapped inside `[e^−ε, e^ε]`, which *is* the ε-DP condition.

In [ ]:
dp_viz.eps_closeness_viz(2.0)   # loose budget (bigger leak): worlds may differ a lot

In [ ]:
dp_viz.eps_closeness_viz(0.5)   # tight budget (small leak): worlds forced close, more noise

### 🧠 Quick checks

In [ ]:
dp_viz.mc_quiz("why_noise")

In [ ]:
dp_viz.mc_quiz("eps_meaning")

---
# Part 4 — The Laplace mechanism: how much noise, exactly?

We have the intuition; now the recipe to make *any* numeric query ε-DP. Before any formula, the
question to ask is simply:

> *How much can the published number move when the one thing I am hiding changes?*

That is the whole motivation. If flipping James's status can only nudge the number by a hair, a hair
of noise already hides him. If it can swing the number wildly, we need proportionally more noise to
keep the same leak budget ε. **Noise needed ∝ how much one person can move the output.**

That "how much" has a name — the **sensitivity** — and once you have it, the amount of noise follows
automatically.

### Warm-up — sampling Laplace noise
Meet the one new tool: **`np.random.default_rng(seed)`** makes a reproducible random generator, and
its **`.laplace(loc, scale)`** draws Laplace noise. `loc` is the centre (0 here) and `scale` is the
`b` from the density `1/(2b)·exp(−|t|/b)`. Draw a few:

In [ ]:
rng = np.random.default_rng(0)
print("five Laplace draws, scale b=2:", rng.laplace(0.0, 2.0, size=5))
print("a bigger scale spreads them out more:", rng.laplace(0.0, 8.0, size=5))

### Sensitivity: the most the hidden thing can move the output
Back to James. The thing we hide there is **his status** (a *defaulter/repayer* flip — an attribute,
not his presence in the dataset). Flipping it moves the published count from 2 to 3: a change of
exactly **1**. So that query's sensitivity is 1, and one unit of "output movement" is what the noise
has to drown. In general:

In [ ]:
dp_viz.sensitivity_viz()

With the sensitivity in hand, the whole mechanism is a one-line theorem. Read the card
carefully — it unpacks the `max`, the norm, and works a full example (publishing an average height)
where you can *see* why a wider data range or a smaller ε both force more noise.

In [ ]:
dp_viz.laplace_theorem_card()

### 🧠 Quick checks

In [ ]:
dp_viz.mc_quiz("sensitivity")

In [ ]:
dp_viz.true_false_quiz("laplace")

---
# Part 5 — Your turn: two Laplace problems

The job is always the same three moves: **① identify the mechanism `f`, ② identify the neighbourhood
and the sensitivity Δ, ③ add `Lap(0, Δ/ε)`.** Here are two, and they are *not* the same shape.

### Problem A — protect an *attribute*, and publish a *mean*
The bank wants to publish the **average height** of its clients (say, for a wellness report).
Heights are known to lie between **140 cm and 220 cm**, there are **N = 200** clients, and the
neighbourhood is *change one client's height* (we're hiding each person's actual height — an
**attribute**, not their membership). This is exactly the worked example from the theorem card.

**Step 1 — find the sensitivity** (no code — reason it out, then type the number):

In [ ]:
dp_viz.number_box("ex1_sensitivity")

**Step 2 — build the private mean.** Now turn that sensitivity into the noisy release.

In [ ]:
rng = np.random.default_rng(5)
heights = rng.uniform(150, 195, size=200)      # 200 clients' true heights (cm)
eps, lo, hi, N = 1.0, 140, 220, len(heights)

sensitivity = (hi - lo) / N                    # Δ₁ for a mean (the number you just found)
scale = ???                                    # 🎯 the Laplace scale from the theorem: Δ₁ / ε
noisy_avg_height = heights.mean() + ???        # 🎯 add one Laplace draw of that scale (loc=0)

print("true average:", round(float(heights.mean()), 2),
      " · private average:", round(float(noisy_avg_height), 2))
assert np.isclose(scale, 0.4), "Δ₁ = 80/200 = 0.4, so scale = 0.4/ε = 0.4 here."
print("✅ a mean over N people has a much smaller sensitivity than a raw sum")

### Problem B — protect *membership*, and publish a 2-D count
Now the bank publishes a **2-D vector**: (number of defaulters in region A, number of defaulters in
region B). This time we want to hide **whether any single client was in the dataset at all** — a
*membership* question. Here is the true statistic, just so it's concrete:

In [ ]:
rng = np.random.default_rng(6)
region = rng.integers(0, 2, size=300)          # each client's region: A (0) or B (1)
defaulted = rng.integers(0, 2, size=300)       # 1 = defaulted
counts_2d = np.array([((region == 0) & (defaulted == 1)).sum(),
                      ((region == 1) & (defaulted == 1)).sum()])
print("true 2-D count (region A defaulters, region B defaulters):", counts_2d)

**Step 1 — which neighbourhood encodes "hide membership"?** (no code)

In [ ]:
dp_viz.mc_quiz("ex2_neighbourhood")

**Step 2 — find the L1 sensitivity Δ₁** of this 2-D count under that neighbourhood. Think
about how many of the two components one added/removed client can change, and by how much (no
code):

In [ ]:
dp_viz.number_box("ex2_sensitivity")

---
# Part 6 — Where ε-DP gets awkward, and how (ε,δ)-DP fixes it

Pure ε-DP has a strict rule hiding in the definition: since the bound is a *ratio*, **no output may
have zero probability under one neighbour while having positive probability under the other** — the
ratio would be infinite. That rules out any noise whose support is "cut off". The picture:

In [ ]:
dp_viz.eps_limit_viz()

The fix is to allow a small **absolute** slack, `δ`. This is **(ε, δ)-DP**.

In [ ]:
dp_viz.eps_delta_card()

### 🧠 Quick check

In [ ]:
dp_viz.mc_quiz("eps_delta")

---
# Part 7 — The Gaussian mechanism *(Optional)*

> ⏭️ **Optional section.** If you are short on time, skip to Part 8 — you can follow the rest with
> just the Laplace mechanism in mind. Come back for the Gaussian when you want the version that
> DP-SGD actually uses.

With δ in hand we can use the friendliest noise of all — the **Gaussian** — which is the one we
need for machine learning (gradients are vectors, and their natural norm is L2).

In [ ]:
dp_viz.gaussian_theorem_card()

The formula ties σ to your budget: **more privacy (smaller ε) needs more noise.**

In [ ]:
dp_viz.gaussian_noise_scale_viz()

### 🎯 The one trap: L2 sensitivity
The Gaussian uses **Δ₂**, the sensitivity in the **L2** norm — the *root of the sum of squares* of
the per-coordinate changes, not their plain sum. It's the same idea as before with one gotcha. Try
it (no code):

In [ ]:
dp_viz.number_box("gauss_l2_sensitivity")

### 🎯 Task — size the Gaussian noise yourself
Same shape of problem as the Laplace ones, but now plug into the Gaussian formula
`σ = √(2 ln(1.25/δ)) · Δ₂ / ε`. We release a single number `f` whose L2 sensitivity is `Δ₂ = 1`.

In [ ]:
rng = np.random.default_rng(3)
eps, delta, sensitivity_l2 = 1.0, 1e-5, 1.0
f_value = 42.0                       # the true statistic we want to release

sigma = ???                          # 🎯 the Gaussian σ from the theorem (use np.log for ln, np.sqrt)
private_value = f_value + rng.normal(0.0, sigma)

print("required σ:", round(float(sigma), 2), " · private release:", round(float(private_value), 2))
assert np.isclose(sigma, np.sqrt(2 * np.log(1.25 / delta)) * sensitivity_l2 / eps)
print("✅ smaller ε or smaller δ would push σ up")

### 🧠 Quick check

In [ ]:
dp_viz.true_false_quiz("gaussian_delta")

---
# Part 8 — Two rules that let you build *big* private systems

So far, one release at a time. Real systems make many. Two properties of DP make this tractable —
and they are the load-bearing beams under DP-SGD.

In [ ]:
dp_viz.composition_diagram()

**Why composition really costs you** — drag the **queries** slider below. Each query is another
noisy release, landing as a bar in the attacker's histogram. A few queries are just noise; with many,
the histogram traces the true distribution and its centre gives away the secret `f(a)`. Every release
hands over a little more — which is why the budget must add up.

In [ ]:
dp_viz.composition_illustration()

### 🧠 Quick check — post-processing or composition?

In [ ]:
dp_viz.true_false_quiz("composability")

---
# Part 9 — A full problem: spending a privacy budget

Now put composition and post-processing together on a real bank scenario, one guided step at a time.

**The setup.**
- **What we protect:** whether *any single client defaulted* — the neighbourhood is *add/remove one
  client*, exactly as in James's story.
- **The total budget:** the bank's policy allows a **total of ε = 1.5** to be disclosed about default
  status. That is the whole budget for everything below.
- **Already spent:** last quarter it published a private **count of defaulters** at **ε₁ = 0.5**.
- **Now:** it wants a *second* release — a private **count of high-risk clients** — and afterwards a
  derived figure (defaulters ÷ high-risk).

(Note: the **number of clients** is public and was known from the start, so releasing it costs
nothing — we never spend budget on it.)

### 🎯 Task — Step 1: neighbourhood and sensitivities
First, name the neighbourhood of these two released numbers, then their sensitivity.

In [ ]:
dp_viz.mc_quiz("full_neighbourhood")

Both releases are plain **counts**. Type the L1 sensitivity **each** of them has under
add/remove-one:

In [ ]:
dp_viz.number_box("full_sensitivity")

### 🎯 Task — Step 2: how much budget is left for the second release?
Composition says the budgets of everything you disclose **add up** and must fit under the total.
Given the total and what's already spent, how much can the second release use?

In [ ]:
dp_viz.number_box("full_eps2")

### Putting it together
With ε₂ settled, here are both private releases and the post-processed ratio — run it and watch the
budget accounting.

In [ ]:
rng = np.random.default_rng(7)
defaulted  = rng.integers(0, 2, size=500)         # 1 = defaulted
high_risk  = rng.integers(0, 2, size=500)         # 1 = flagged high-risk
eps_total, eps1, eps2 = 1.5, 0.5, 1.0             # eps2 = eps_total - eps1  (Step 2)

release_defaulters = defaulted.sum() + rng.laplace(0.0, 1 / eps1)   # Δ₁=1 → scale 1/ε₁
release_highrisk   = high_risk.sum() + rng.laplace(0.0, 1 / eps2)   # Δ₁=1 → scale 1/ε₂
ratio = release_defaulters / release_highrisk      # post-processing of private outputs → FREE

print("private #defaulters :", round(float(release_defaulters), 1), "(spent ε₁ = 0.5)")
print("private #high-risk  :", round(float(release_highrisk),   1), "(spent ε₂ = 1.0)")
print("derived ratio       :", round(float(ratio), 3), "(post-processing → 0 extra budget)")
print("TOTAL spent ε        :", eps1 + eps2, "≤", eps_total, "✅ within budget")

### 🧠 Quick check

In [ ]:
dp_viz.mc_quiz("composition")

---
# Part 10 — The payoff: DP-SGD

Everything so far was building to this: a **private training algorithm**. But *how* to make training
private isn't obvious — so let's reason it out.

Our setting is the one from the very start of this notebook: the bank's classifier predicting whether
a client will **default on a loan**. Concretely it is the same logistic regression you met in
notebook 03, re-dressed for lending:

- it reads **two client features** — the *debt-to-income ratio* (`x₁`) and the *number of missed
  payments in the past year* (`x₂`);
- it computes a score `z = w·x + b`, then squashes it into a probability with `σ(z) = 1/(1+e^(−z))`;
- it predicts **default if `σ(z) ≥ 0.5`, else repay** — and `σ(z) ≥ 0.5` is just `z ≥ 0`.

Both features push the same way: more debt and more missed payments mean a higher score, so a higher
predicted chance of default. And because the decision only depends on the sign of `z`, the model *is*
a straight line in the feature plane — "the model" simply means the weights `w, b` that place it.

### Recap 1 — our model is a decision boundary
Here is that classifier: the sigmoid on the left, and on the right the line it draws through the two
loan features, splitting clients into **default** and **repay**.

In [ ]:
w0, b0 = np.array([1.0, 1.0]), 0.0
dp_viz.logreg_recap(w=w0, b=b0)

### Recap 2 — how it's trained
We train for **T** epochs over **N** clients. To keep the analysis clean we take **batch size 1**
(one client per step): each step nudges `w, b` a little using that client's gradient.

### Where do we even add the noise?
It's tempting to treat the *finished model's output* as the mechanism and noise that. Reason through
the options — each fails until the last:
- **Noise the 0/1 decision?** Flipping labels is crude and wrecks accuracy.
- **Noise the probability?** It's forced outside [0, 1], and one client can swing it the full 0→1 —
  a terrible sensitivity.
- **Noise the logits?** Maybe — but bounding a logit's sensitivity is very hard (it depends on the
  features and the model), so you either can't prove a bound or must assume a uselessly large one.

And even a good logit bound has a killer flaw at *inference* time:

In [ ]:
dp_viz.mc_quiz("query_composition")

So inference-time noise is a trap. The clean answer: a client's data only ever touches the
model **during training, at the gradient step.** So we make *that* step private — treat each gradient
update as a **private release of gradient information** — with the two moves you already know: **clip**
its norm (bound the sensitivity) and **add Gaussian noise**.

In [ ]:
dp_viz.mc_quiz("noise_where")

Here is the full DP-SGD loop — every piece is something you've already met:

In [ ]:
dp_viz.dpsgd_diagram()

The clip caps each person's gradient to a fixed length `C`, which is exactly how we *bound the
sensitivity*:

In [ ]:
C = 1.0
g = np.array([3.0, 4.0])               # a gradient with L2 norm 5
clipped = g / max(1.0, np.linalg.norm(g) / C)
print("norm before:", np.linalg.norm(g), " · norm after clip:", round(float(np.linalg.norm(clipped)), 3))
# a gradient already shorter than C is left untouched:
small = np.array([0.2, 0.1])
print("short gradient stays put:", small / max(1.0, np.linalg.norm(small) / C))

> 🧰 **NumPy tools for this task** (one-liners, so you don't have to guess the syntax):
> - `np.linalg.norm(a, axis=1, keepdims=True)` — the L2 length of **each row**, kept as a column so
>   it divides back cleanly (row-wise).
> - `np.maximum(1.0, x)` — element-wise max against 1 (short gradients keep their norm; long ones get
>   scaled down). Different from `np.max`, which collapses to a single number.
> - `a.sum(axis=0)` — sum **down the rows** → one vector, one entry per column.
> - `rng.normal(0.0, sigma, size=3)` — three independent Gaussian draws (mean 0, std `sigma`).

### 🎯 Task — one private (DP-SGD) update step
You are given the **per-example gradients** for a mini-batch — one row per client. Turn them into a
private aggregate: **clip each row to norm `C`, sum, then add Gaussian noise**.

In [ ]:
rng = np.random.default_rng(4)
grads = rng.normal(0, 1.5, size=(8, 3))    # 8 clients, a 3-dim gradient each
C, sigma = 1.0, 4.0

norms = np.linalg.norm(grads, axis=1, keepdims=True)
clipped = grads / np.maximum(1.0, norms / C)   # 🎯 (given) clip each ROW to L2 norm ≤ C
summed = ???       # 🎯 aggregate the clipped per-example gradients (sum over the 8 clients / rows)
private_grad = ??? # 🎯 add Gaussian noise: summed + one N(0, σ) draw per coordinate (size 3)

# self-check: every clipped row now has norm ≤ C, so the sensitivity is bounded
assert np.all(np.linalg.norm(clipped, axis=1) <= C + 1e-9)
assert private_grad.shape == (3,)
print("private aggregated gradient:", np.round(private_grad, 2))
print("✅ clip (bound sensitivity) + Gaussian noise = one private gradient step")

### 🎯 The privacy budget of the whole run
Each step is one Gaussian-mechanism release. Over **T** steps, what does the whole training cost?

In [ ]:
dp_viz.mc_quiz("dpsgd_budget")

By post-processing, the final trained model then carries that *same* budget — you can serve it
and take predictions for free.

### 🎯 Two sensitivity variants (batch size L)
Where you inject the noise changes the sensitivity — and therefore the noise you need. Reason out
each (no code).

**Variant 1 — noise the SUM** of a batch of L clipped gradients:

In [ ]:
dp_viz.number_box("dpsgd_sum_sensitivity")

**Variant 2 — noise the MEAN** (divide the sum by L *before* adding noise):

In [ ]:
dp_viz.number_box("dpsgd_mean_sensitivity")

Same signal, smaller footprint: averaging divides the sensitivity by `L`, so the mean needs
proportionally less noise for the same ε. Either way, the **total** budget over T steps follows from
**composition** — the per-step ε's add up.

### Does it still learn? Yes — a little blurrier
Below, the **same** bank classifier trained normally vs with DP-SGD. The private boundary is close
but perturbed: that wobble is the price that hides any single client.

In [ ]:
dp_viz.dpsgd_boundary_viz()

### The one trade-off you can never escape
More noise buys more privacy (smaller ε) but costs accuracy. There is **no free lunch** — this curve
is the single most important picture in all of DP, and it holds for *every* DP method, not just
DP-SGD.

In [ ]:
dp_viz.utility_privacy_curve()

### 🧠 Quick check

In [ ]:
dp_viz.true_false_quiz("dpsgd")

### Where this goes next
DP-SGD is the standard, but not the only route to private ML — and the same building blocks power
all of them:
- **PATE** — train many "teacher" models on disjoint slices, then release only their **noisy vote**
  (a Laplace/Gaussian mechanism + composition + post-processing). Often better accuracy per ε.
- **Federated learning + DP** — run DP-SGD on each client's device and add noise before the server
  aggregates, so raw data never leaves the phone.
- **Randomized smoothing** — the *same* noise-plus-post-processing trick, repurposed to certify a
  model's robustness to adversarial inputs (a neat bridge to the robustness lectures).


---
# 🎓 Wrap-up — the private-ML checklist

You went from "a count can betray one person" to a full private training loop, using one idea over
and over: **make the output barely depend on any single individual.**

| Building block | What it does | When you reach for it |
|---|---|---|
| **Neighbourhood** | defines *what you hide* (e.g. one person in/out) | always — pick it first |
| **Sensitivity Δ** | how much one person can move the output | to size the noise |
| **Laplace mechanism** | `f + Lap(0, Δ₁/ε)` → ε-DP | scalar / counting queries |
| **Gaussian mechanism** | `f + 𝒩(0, σ²)` → (ε,δ)-DP | vector outputs, gradients |
| **Post-processing** | anything computed from a DP output stays DP | deriving results — free |
| **Composition** | budgets add across releases | many queries, training loops |
| **DP-SGD** | clip + noise every gradient step | training a private model |

**The one-line manager takeaway:** *privacy is a budget you spend.* Every number you release about
people costs some of it; noise is how you pay; and ε is the price you're willing to pay to let
individuals disappear into the crowd.

---
## 🏁 Final boss — clear the notebook
Everything you just learned, one statement at a time: **membership inference, the ε definition,
neighbourhoods, sensitivity, Laplace & Gaussian, (ε,δ), composition, post-processing, DP-SGD.** The
rules: **3 lives**, **10 seconds** per question, and a wrong answer *or* a timeout costs a life.
Reach **10 correct** to pass. Good luck. 🍀

In [ ]:
dp_viz.flash_quiz()